# 🧠 GlassBox Quickstart Guide

This notebook demonstrates the core functionality of GlassBox:
1. Capturing attention patterns from GPT-2
2. Analyzing which attention heads contribute to outputs
3. Computing token-level influence scores
4. Saving traces as structured JSON artifacts

**Runtime**: ~5-10 minutes on CPU

## Setup

In [ ]:
# Imports
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from glassbox_tracer import ActivationTracer, TracerConfig
from glassbox_analyzer import AttentionAnalyzer
from glassbox_serializer import TraceSerializer

import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful")

## 1. Initialize Components

In [ ]:
# Initialize tracer (downloads GPT-2-small on first run)
print("Loading GPT-2-small...")
tracer = ActivationTracer(model_name="gpt2-small")

# Initialize analyzer and serializer
analyzer = AttentionAnalyzer()
serializer = TraceSerializer(output_dir="../data/traces")

print("✅ Components initialized")

## 2. Basic Trace: Simple Question

In [ ]:
# Simple prompt
prompt = "The capital of France is"

# Run trace (captures all layers)
print(f"Tracing: '{prompt}'")
result = tracer.trace(prompt)

print(f"\n📊 Results:")
print(f"  Input: {result.prompt}")
print(f"  Output: {result.output_text}")
print(f"  Tokens: {len(result.tokens)}")
print(f"  Inference time: {result.metadata.inference_time_ms:.1f}ms")
print(f"  Slowdown: {result.metadata.slowdown_factor:.2f}x")
print(f"  Captured heads: {len(result.attention_cache)}")

## 3. Analyze Attention Patterns

In [ ]:
# Rank attention heads by contribution
top_heads = analyzer.rank_attention_heads(
    result.attention_cache,
    result.tokens,
    target_token_idx=-1  # Last token (output)
)

print("🎯 Top 5 Contributing Attention Heads:\n")
for i, head in enumerate(top_heads[:5], 1):
    print(f"{i}. Layer {head.layer}, Head {head.head}")
    print(f"   Score: {head.score:.3f}")
    print(f"   Top attended tokens:")
    for token, weight in head.top_attended_tokens[:3]:
        print(f"     → '{token}': {weight:.3f}")
    print()

## 4. Token-Level Influence

In [ ]:
# Compute aggregate influence per token
token_influence = analyzer.compute_token_influence(
    result.attention_cache,
    result.tokens,
    target_token_idx=-1,
    top_n_heads=10
)

print("🌡️ Token Influence Scores:\n")
top_tokens = analyzer.get_top_contributing_tokens(token_influence, top_k=5)

for token, score in top_tokens:
    bar = "█" * int(score * 30)
    print(f"  '{token:15s}' {bar} {score:.3f}")

## 5. Advanced Example: Financial Decision

In [ ]:
# More complex prompt
financial_prompt = "Should we approve this loan application? Credit score: 750, Income: $85k"

# Focus on middle layers (where semantic processing happens)
config = TracerConfig(
    capture_layers=[5, 6, 7, 8, 9, 10],
    max_seq_length=128
)

print(f"Tracing financial decision...")
fin_result = tracer.trace(financial_prompt, config)

print(f"\n📊 Financial Trace Results:")
print(f"  Input: {fin_result.prompt}")
print(f"  Output: {fin_result.output_text}")
print(f"  Time: {fin_result.metadata.inference_time_ms:.0f}ms")

In [ ]:
# Analyze financial decision
fin_heads = analyzer.rank_attention_heads(
    fin_result.attention_cache,
    fin_result.tokens
)

print("🎯 Top Heads for Financial Decision:\n")
for i, head in enumerate(fin_heads[:3], 1):
    print(f"{i}. Layer {head.layer}, Head {head.head} (score: {head.score:.3f})")
    print("   Attended to:")
    for token, weight in head.top_attended_tokens[:5]:
        print(f"     → '{token}': {weight:.3f}")
    print()

In [ ]:
# Token influence for financial decision
fin_influence = analyzer.compute_token_influence(
    fin_result.attention_cache,
    fin_result.tokens,
    top_n_heads=10
)

print("🌡️ Token Influence on Financial Decision:\n")
fin_top = analyzer.get_top_contributing_tokens(fin_influence, top_k=10)

for token, score in fin_top:
    bar = "█" * int(score * 25)
    print(f"  '{token:20s}' {bar} {score:.3f}")

## 6. Save Trace as JSON

In [ ]:
# Save the financial trace
filepath = serializer.save(
    fin_result,
    top_n_heads=10,
    git_commit="demo_notebook"
)

print(f"\n✅ Trace saved to: {filepath}")

# Get trace ID
trace_id = filepath.stem.replace("trace_", "")
print(f"   Trace ID: {trace_id}")

## 7. Load and Inspect Saved Trace

In [ ]:
# Load the trace back
loaded_trace = serializer.load(trace_id)

print("📄 Loaded Trace Structure:\n")
print(f"  Trace ID: {loaded_trace['trace_id']}")
print(f"  Model: {loaded_trace['model']}")
print(f"  Input: {loaded_trace['input']['text']}")
print(f"  Output: {loaded_trace['output']['text']}")
print(f"  Confidence: {loaded_trace['output']['probability']:.2%}")
print(f"  Top heads captured: {len(loaded_trace['attribution']['top_attention_heads'])}")
print(f"  Token influences: {len(loaded_trace['attribution']['token_influence'])}")

In [ ]:
# Display JSON structure (pretty print)
import json

print("\n📋 Full JSON Structure (first head only):\n")
preview = {
    "trace_id": loaded_trace['trace_id'],
    "input": loaded_trace['input'],
    "output": loaded_trace['output'],
    "top_head": loaded_trace['attribution']['top_attention_heads'][0],
    "performance": loaded_trace['performance']
}

print(json.dumps(preview, indent=2))

## 8. Multiple Traces Comparison

In [ ]:
# Create multiple traces for comparison
prompts = [
    "The weather today is",
    "Paris is the capital of",
    "Machine learning is"
]

traces = []
for prompt in prompts:
    result = tracer.trace(prompt, TracerConfig(capture_layers=[8, 9, 10]))
    traces.append({
        'prompt': prompt,
        'output': result.output_text,
        'time': result.metadata.inference_time_ms,
        'result': result
    })

print("📊 Batch Trace Results:\n")
for i, t in enumerate(traces, 1):
    print(f"{i}. '{t['prompt']}' → '{t['output']}'")
    print(f"   Time: {t['time']:.0f}ms\n")

## 9. List All Traces

In [ ]:
# List all saved traces
all_traces = serializer.list_traces(limit=20)

print(f"📚 Found {len(all_traces)} saved trace(s):\n")

for trace in all_traces[:10]:  # Show first 10
    print(f"  ID: {trace['trace_id']}")
    print(f"  Time: {trace['timestamp']}")
    print(f"  Input: {trace['input_preview'][:60]}...")
    print(f"  Output: {trace['output']}")
    print()

## 10. Next Steps

Now that you've completed the quickstart:

1. **Launch the Dashboard**:
   ```bash
   streamlit run dashboard/app.py
   ```

2. **Try the API**:
   ```bash
   uvicorn api.server:app --reload --port 8000
   ```

3. **Explore Advanced Notebooks**:
   - `02_custom_analysis.ipynb` - Custom attribution methods
   - `03_validation_tests.ipynb` - Interpretability validation

4. **Read the Documentation**:
   - Architecture overview
   - API reference
   - Validation methodology

---

**🎉 You've successfully traced and analyzed GPT-2 internals with GlassBox!**